# Chapter 3 — Model Training

### Chapter Overview

This chapter trains three machine learning models for binary claim prediction.
Each model represents a different methodological approach, from interpretable
statistical baseline to advanced ensemble methods, enabling a meaningful
performance comparison in Chapter 4.

| Section | Activity |
|---------|----------|
| 3.1 | Load cleaned data and imports |
| 3.2 | Model 1 — Logistic Regression (statistical baseline) |
| 3.3 | Model 2 — Random Forest |
| 3.4 | Model 3 — XGBoost |
| 3.5 | Save all trained models to the models/ folder |

### Modelling Decisions

| Decision | Choice | Justification |
|----------|--------|---------------|
| Class imbalance | class_weight='balanced' | Dataset has 26:1 ratio — adjusts the loss function to penalise missed claims more heavily |
| Baseline model | Logistic Regression | Interpretable statistical baseline consistent with the actuarial GLM tradition |
| Ensemble models | Random Forest + XGBoost | Capture non-linear relationships and feature interactions identified in Chapter 2 |
| Parameter selection | Justified defaults | Parameters chosen based on established literature and dataset characteristics rather than exhaustive search |
| Evaluation metric | AUC-ROC | Most appropriate metric for imbalanced binary classification |
| Model persistence | joblib .pkl | Trained models saved for direct loading in Chapters 4 and 5 |

> **Note on Hyperparameter Tuning:** Given the dataset size of 595,212 records,
> exhaustive GridSearchCV would require several hours of computation. Parameters
> are instead selected based on published best practices for insurance datasets
> of this scale, specifically Breiman (2001) for Random Forest and
> Chen & Guestrin (2016) for XGBoost. This approach is standard in
> applied actuarial machine learning research.

> **Research connection:** Model choices directly address RQ1 from Chapter 0,
> which algorithm performs best for claim prediction on an imbalanced dataset?

---


# ── 3.1 IMPORTS AND LOAD ─────────────────────────────────────
### 3.1 Import Libraries and Load Data

Cleaned and split data is loaded from the compressed files saved by Chapter 1.
Feature scaling is applied here for Logistic Regression only. Random Forest
and XGBoost are tree-based models that split on feature thresholds, they are
not sensitive to feature scale and do not require standardisation.


In [1]:
# ── Standard libraries ───────────────────────────────────────
import numpy as np
import pandas as pd
import os
import sys
import joblib                           # Save and load trained models
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn — models ──────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ── Sklearn — validation ──────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ── XGBoost ───────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Evaluation preview ────────────────────────────────────────
from sklearn.metrics import roc_auc_score

# ── Import from src ───────────────────────────────────────────
sys.path.append('../src')
from data_loader import load_cleaned_data, scale_features

print(" Libraries imported successfully.")


 Libraries imported successfully.


In [2]:
# ── Load cleaned data saved by Chapter 1 ─────────────────────
data = load_cleaned_data('../data')

X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']

# ── Scale features for Logistic Regression only ───────────────
# Fit scaler on training data only — prevents data leakage
# RF and XGBoost use unscaled X_train / X_test directly
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

print(f"\\n Data loaded and ready.")
print(f"   Training samples : {X_train.shape[0]:,}")
print(f"   Test samples     : {X_test.shape[0]:,}")
print(f"   Features         : {X_train.shape[1]}")
print(f"   Claim rate       : {y_train.mean()*100:.2f}%")


 Loading cleaned data from data folder...
 Cleaned data loaded.
   Full dataset : (595212, 206)
   X_train      : (476169, 205)
   X_test       : (119043, 205)
   y_train      : (476169,)
   y_test       : (119043,)
 Scaling complete.
   Fit on training data only (no data leakage)
\n Data loaded and ready.
   Training samples : 476,169
   Test samples     : 119,043
   Features         : 205
   Claim rate       : 3.64%


In [3]:
# ── Set up paths and cross-validation ────────────────────────
# Models folder — one level up from notebooks/
MODELS_PATH = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

# StratifiedKFold — preserves the ~3.6% claim rate in each fold
# Essential for reliable CV scores on imbalanced data
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f" Models folder    : {MODELS_PATH}")
print(f" CV strategy      : StratifiedKFold (5 folds, stratified)")


 Models folder    : C:\Users\Administrator\Desktop\policyholder-risk-predictions\models
 CV strategy      : StratifiedKFold (5 folds, stratified)


# ── 3.2 LOGISTIC REGRESSION ──────────────────────────────────
### 3.2 Model 1 — Logistic Regression (Statistical Baseline)

Logistic Regression models the log-odds of a claim as a linear combination
of policyholder features. It is the machine learning equivalent of the
Generalised Linear Model (GLM) that underpins traditional actuarial pricing.

Including Logistic Regression as a baseline serves a specific research purpose,
it allows Chapter 4 to quantify exactly how much predictive performance is
gained by moving to more complex ensemble methods. This is a meaningful
contribution in an actuarial context where model complexity must be justified.

**Parameter justification:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| C | 1.0 | Default L2 regularisation — prevents overfitting on the large feature space without strong prior assumption on penalty strength |
| solver | lbfgs | Memory-efficient solver recommended for large datasets with L2 regularisation |
| max_iter | 1000 | Sufficient iterations to guarantee convergence on 476,000 training samples |
| class_weight | balanced | Automatically adjusts weights inversely proportional to class frequencies — corrects the 26:1 imbalance |


In [4]:
# ── Train Logistic Regression ────────────────────────────────
print(" Training Logistic Regression...")

lr_model = LogisticRegression(
    C=1.0,                      # L2 regularisation strength
    solver='lbfgs',             # Efficient for large datasets + L2
    max_iter=1000,              # Ensures convergence
    class_weight='balanced',    # Corrects 26:1 class imbalance
    random_state=42,
    n_jobs=-1                   # Use all CPU cores
)

lr_model.fit(X_train_scaled, y_train)

# ── Preview performance ───────────────────────────────────────
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc        = roc_auc_score(y_test, lr_pred_proba)

print(f"\\n Logistic Regression trained successfully.")
print(f"   Test AUC-ROC    : {lr_auc:.4f}")
print(f"   n_iter_         : {lr_model.n_iter_[0]} (converged)")
print(f"   n_features      : {lr_model.coef_.shape[1]}")


 Training Logistic Regression...
\n Logistic Regression trained successfully.
   Test AUC-ROC    : 0.6272
   n_iter_         : 47 (converged)
   n_features      : 205


In [5]:
# ── Cross-validation — Logistic Regression ───────────────────
# 5-fold CV on training data gives a more reliable estimate
# of generalisation performance than a single train/test split
print(" Running 5-fold cross-validation for Logistic Regression...")

lr_cv = cross_val_score(
    LogisticRegression(
        C=1.0, solver='lbfgs', max_iter=1000,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    X_train_scaled, y_train,
    cv=CV,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\\n Logistic Regression — 5-Fold CV Results:")
print(f"   Fold AUC scores  : {[round(s,4) for s in lr_cv]}")
print(f"   Mean AUC         : {lr_cv.mean():.4f}")
print(f"   Std deviation    : {lr_cv.std():.4f}")
print(f"   Interpretation   : {'Stable ' if lr_cv.std() < 0.01 else 'Some variance — review folds'}")


 Running 5-fold cross-validation for Logistic Regression...
\n Logistic Regression — 5-Fold CV Results:
   Fold AUC scores  : [np.float64(0.6268), np.float64(0.6286), np.float64(0.6201), np.float64(0.6209), np.float64(0.6273)]
   Mean AUC         : 0.6247
   Std deviation    : 0.0035
   Interpretation   : Stable 


# ── 3.3 RANDOM FOREST ────────────────────────────────────────
### 3.3 Model 2 — Random Forest

Random Forest is a bagging ensemble that trains multiple independent decision
trees on random subsets of rows and features, then aggregates their predictions
by majority vote. It naturally handles non-linear relationships and feature
interactions, and is robust to outliers properties well suited to insurance
policyholder data.

**Parameter justification:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| n_estimators | 200 | 200 trees provides stable predictions — Breiman (2001) shows performance plateaus beyond 100–200 trees on most datasets |
| max_depth | 15 | Prevents overfitting while allowing trees enough depth to capture complex risk interactions |
| min_samples_split | 10 | Requires at least 10 samples to split a node — reduces noise fitting on the large dataset |
| min_samples_leaf | 4 | Minimum 4 samples per leaf — smooths predictions and prevents memorising rare observations |
| max_features | sqrt | Square root of features per split — standard recommendation from Breiman (2001) for classification tasks |
| class_weight | balanced | Corrects the 26:1 class imbalance by weighting minority class inversely to its frequency |


In [6]:
# ── Train Random Forest ──────────────────────────────────────
print(" Training Random Forest...")
print("   Estimated time: 3-8 minutes on this dataset size...")

rf_model = RandomForestClassifier(
    n_estimators=200,           # 200 trees — stable ensemble
    max_depth=15,               # Prevents overfitting
    min_samples_split=10,       # Min samples to split a node
    min_samples_leaf=4,         # Min samples per leaf node
    max_features='sqrt',        # sqrt(n_features) per split
    class_weight='balanced',    # Corrects 26:1 imbalance
    random_state=42,
    n_jobs=-1                   # Use all CPU cores
)

rf_model.fit(X_train, y_train)  # No scaling needed for trees

# ── Preview performance ───────────────────────────────────────
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc        = roc_auc_score(y_test, rf_pred_proba)

print(f"\\n Random Forest trained successfully.")
print(f"   Test AUC-ROC    : {rf_auc:.4f}")
print(f"   n_estimators    : {rf_model.n_estimators}")
print(f"   n_features_in   : {rf_model.n_features_in_}")


 Training Random Forest...
   Estimated time: 3-8 minutes on this dataset size...
\n Random Forest trained successfully.
   Test AUC-ROC    : 0.6076
   n_estimators    : 200
   n_features_in   : 205


In [7]:
# ── Train Random Forest ──────────────────────────────────────
print(" Training Random Forest...")
print("   Estimated time: 3-8 minutes on this dataset size...")

rf_model = RandomForestClassifier(
    n_estimators=200,           # 200 trees — stable ensemble
    max_depth=15,               # Prevents overfitting
    min_samples_split=10,       # Min samples to split a node
    min_samples_leaf=4,         # Min samples per leaf node
    max_features='sqrt',        # sqrt(n_features) per split
    class_weight='balanced',    # Corrects 26:1 imbalance
    random_state=42,
    n_jobs=-1                   # Use all CPU cores
)

rf_model.fit(X_train, y_train)  # No scaling needed for trees

# ── Preview performance ───────────────────────────────────────
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc        = roc_auc_score(y_test, rf_pred_proba)

print(f"\\n Random Forest trained successfully.")
print(f"   Test AUC-ROC    : {rf_auc:.4f}")
print(f"   n_estimators    : {rf_model.n_estimators}")
print(f"   n_features_in   : {rf_model.n_features_in_}")


 Training Random Forest...
   Estimated time: 3-8 minutes on this dataset size...
\n Random Forest trained successfully.
   Test AUC-ROC    : 0.6076
   n_estimators    : 200
   n_features_in   : 205


In [8]:
# ── Cross-validation — Random Forest ─────────────────────────
print(" Running 5-fold cross-validation for Random Forest...")

rf_cv = cross_val_score(
    RandomForestClassifier(
        n_estimators=200, max_depth=15,
        min_samples_split=10, min_samples_leaf=4,
        max_features='sqrt', class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    X_train, y_train,
    cv=CV,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\\n Random Forest — 5-Fold CV Results:")
print(f"   Fold AUC scores  : {[round(s,4) for s in rf_cv]}")
print(f"   Mean AUC         : {rf_cv.mean():.4f}")
print(f"   Std deviation    : {rf_cv.std():.4f}")
print(f"   Interpretation   : {'Stable ' if rf_cv.std() < 0.01 else 'Some variance — review folds'}")


 Running 5-fold cross-validation for Random Forest...
\n Random Forest — 5-Fold CV Results:
   Fold AUC scores  : [np.float64(0.5982), np.float64(0.6061), np.float64(0.5954), np.float64(0.6014), np.float64(0.6049)]
   Mean AUC         : 0.6012
   Std deviation    : 0.0040
   Interpretation   : Stable 


# ── 3.4 XGBOOST ──────────────────────────────────────────────
### 3.4 Model 3 — XGBoost

XGBoost is a gradient boosting algorithm that builds trees sequentially,
each new tree is trained to correct the residual errors of the previous
ensemble. It is the dominant algorithm on structured tabular data benchmarks
and achieved top performance in the original Porto Seguro Kaggle competition,
making it a natural and well justified choice for this dataset.

**Handling class imbalance in XGBoost:**
XGBoost uses `scale_pos_weight` instead of `class_weight`. Set to the ratio
of negative to positive samples, it scales the gradient contribution of
minority class samples, equivalent to oversampling the minority class
during training without generating synthetic data.

**Parameter justification:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| n_estimators | 200 | 200 boosting rounds — balances performance and training time at this dataset scale |
| max_depth | 6 | Standard depth for tabular insurance data — Chen & Guestrin (2016) recommend 4–8 |
| learning_rate | 0.05 | Lower learning rate with more trees — produces more robust models than high learning rate with fewer trees |
| subsample | 0.8 | 80% row sampling per tree — reduces overfitting and adds stochastic regularisation |
| colsample_bytree | 0.8 | 80% feature sampling per tree — consistent with Random Forest max_features rationale |
| scale_pos_weight | neg/pos | Computed from training class counts — corrects the 26:1 imbalance natively in XGBoost |


In [9]:
# ── Compute scale_pos_weight ──────────────────────────────────
# Ratio of negative to positive class in training set
# Tells XGBoost to up-weight the minority class proportionally
neg_count        = (y_train == 0).sum()
pos_count        = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"  XGBoost class weight:")
print(f"   No claim (0) : {neg_count:,}")
print(f"   Claim    (1) : {pos_count:,}")
print(f"   scale_pos_weight = {scale_pos_weight:.2f}")


  XGBoost class weight:
   No claim (0) : 458,814
   Claim    (1) : 17,355
   scale_pos_weight = 26.44


In [10]:
# ── Train XGBoost ─────────────────────────────────────────────
print("\\n Training XGBoost...")
print("   Estimated time: 3-8 minutes on this dataset size...")

xgb_model = XGBClassifier(
    n_estimators=200,                   # Boosting rounds
    max_depth=6,                        # Tree depth per round
    learning_rate=0.05,                 # Step size — lower = more robust
    subsample=0.8,                      # 80% row sampling per tree
    colsample_bytree=0.8,               # 80% feature sampling per tree
    scale_pos_weight=scale_pos_weight,  # Corrects class imbalance
    eval_metric='auc',                  # Internal evaluation metric
    random_state=42,
    n_jobs=-1,
    verbosity=0                         # Suppress XGBoost internal logs
)

xgb_model.fit(X_train, y_train)

# ── Preview performance ───────────────────────────────────────
xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc        = roc_auc_score(y_test, xgb_pred_proba)

print(f"\\n XGBoost trained successfully.")
print(f"   Test AUC-ROC    : {xgb_auc:.4f}")
print(f"   n_estimators    : {xgb_model.n_estimators}")
print(f"   n_features      : {xgb_model.n_features_in_}")


\n Training XGBoost...
   Estimated time: 3-8 minutes on this dataset size...
\n XGBoost trained successfully.
   Test AUC-ROC    : 0.6343
   n_estimators    : 200
   n_features      : 205


In [11]:
# ── Cross-validation — XGBoost ───────────────────────────────
print(" Running 5-fold cross-validation for XGBoost...")

xgb_cv = cross_val_score(
    XGBClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc', random_state=42,
        n_jobs=-1, verbosity=0
    ),
    X_train, y_train,
    cv=CV,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\\n XGBoost — 5-Fold CV Results:")
print(f"   Fold AUC scores  : {[round(s,4) for s in xgb_cv]}")
print(f"   Mean AUC         : {xgb_cv.mean():.4f}")
print(f"   Std deviation    : {xgb_cv.std():.4f}")
print(f"   Interpretation   : {'Stable ' if xgb_cv.std() < 0.01 else 'Some variance — review folds'}")


 Running 5-fold cross-validation for XGBoost...
\n XGBoost — 5-Fold CV Results:
   Fold AUC scores  : [np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]
   Mean AUC         : nan
   Std deviation    : nan
   Interpretation   : Some variance — review folds


# ── 3.5 SAVE MODELS ──────────────────────────────────────────
### 3.5 Save Trained Models

All three trained models and the fitted scaler are saved to the `models/`
folder using `joblib`. Chapters 4 and 5 load these files directly
avoiding the need to retrain models which would take several minutes each
time a later chapter is opened and run.

The parameter choices for each model are also saved as a human-readable
CSV, making the methodology transparent and auditable on GitHub.


In [12]:
# ── Save models and scaler ────────────────────────────────────
print(" Saving all trained models...")

# Logistic Regression
lr_path = os.path.join(MODELS_PATH, 'logistic_regression.pkl')
joblib.dump(lr_model, lr_path)
print(f" Logistic Regression → {lr_path}")

# Random Forest
rf_path = os.path.join(MODELS_PATH, 'random_forest.pkl')
joblib.dump(rf_model, rf_path)
print(f" Random Forest       → {rf_path}")

# XGBoost
xgb_path = os.path.join(MODELS_PATH, 'xgboost.pkl')
joblib.dump(xgb_model, xgb_path)
print(f" XGBoost             → {xgb_path}")

# StandardScaler — must be saved for consistent LR preprocessing
scaler_path = os.path.join(MODELS_PATH, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f" Scaler              → {scaler_path}")


 Saving all trained models...
 Logistic Regression → C:\Users\Administrator\Desktop\policyholder-risk-predictions\models\logistic_regression.pkl
 Random Forest       → C:\Users\Administrator\Desktop\policyholder-risk-predictions\models\random_forest.pkl
 XGBoost             → C:\Users\Administrator\Desktop\policyholder-risk-predictions\models\xgboost.pkl
 Scaler              → C:\Users\Administrator\Desktop\policyholder-risk-predictions\models\scaler.pkl


In [14]:
# ── Save model parameters as human-readable CSV ───────────────
params_summary = pd.DataFrame([
    {
        'Model'         : 'Logistic Regression',
        'Key Parameters': 'C=1.0, solver=lbfgs, max_iter=1000',
        'Imbalance Fix' : 'class_weight=balanced',
        'CV AUC Mean'   : round(lr_cv.mean(), 4),
        'CV AUC Std'    : round(lr_cv.std(), 4)
    },
    {
        'Model'         : 'Random Forest',
        'Key Parameters': 'n_estimators=200, max_depth=15, min_samples_split=10',
        'Imbalance Fix' : 'class_weight=balanced',
        'CV AUC Mean'   : round(rf_cv.mean(), 4),
        'CV AUC Std'    : round(rf_cv.std(), 4)
    },
    {
        'Model'         : 'XGBoost',
        'Key Parameters': 'n_estimators=200, max_depth=6, learning_rate=0.05',
        'Imbalance Fix' : f'scale_pos_weight={scale_pos_weight:.2f}',
        'CV AUC Mean'   : round(xgb_cv.mean(), 4),
        'CV AUC Std'    : round(xgb_cv.std(), 4)
    }
])

params_path = os.path.join(MODELS_PATH, 'model_parameters.csv')
params_summary.to_csv(params_path, index=False)

print(f"\\n Parameters saved → {params_path}")
print("\\n Model Parameters Summary:")
print(params_summary.to_string(index=False))


\n Parameters saved → C:\Users\Administrator\Desktop\policyholder-risk-predictions\models\model_parameters.csv
\n Model Parameters Summary:
              Model                                       Key Parameters          Imbalance Fix  CV AUC Mean  CV AUC Std
Logistic Regression                   C=1.0, solver=lbfgs, max_iter=1000  class_weight=balanced       0.6247      0.0035
      Random Forest n_estimators=200, max_depth=15, min_samples_split=10  class_weight=balanced       0.6012      0.0040
            XGBoost    n_estimators=200, max_depth=6, learning_rate=0.05 scale_pos_weight=26.44          NaN         NaN


In [16]:
# ── Chapter 3 summary ────────────────────────────────────────
best_model = max(
    [('Logistic Regression', lr_auc),
     ('Random Forest',       rf_auc),
     ('XGBoost',             xgb_auc)],
    key=lambda x: x[1]
)

print("=" * 60)
print("  CHAPTER 3 COMPLETE — MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"""
  Models trained         : 3
  ┌─────────────────────────────────────────┐
  │ Model                │ Test AUC-ROC     │
  ├─────────────────────────────────────────┤
  │ Logistic Regression  │ {lr_auc:.4f}           │
  │ Random Forest        │ {rf_auc:.4f}           │
  │ XGBoost              │ {xgb_auc:.4f}           │
  └─────────────────────────────────────────┘

  Best performing model  : {best_model[0]} (AUC = {best_model[1]:.4f})
  Class imbalance fix    : class_weight='balanced' / scale_pos_weight
  Models saved to        : models/ folder
""")
print("  Proceed to Chapter 4 — Model Evaluation")

  CHAPTER 3 COMPLETE — MODEL TRAINING SUMMARY

  Models trained         : 3
  ┌─────────────────────────────────────────┐
  │ Model                │ Test AUC-ROC     │
  ├─────────────────────────────────────────┤
  │ Logistic Regression  │ 0.6272           │
  │ Random Forest        │ 0.6076           │
  │ XGBoost              │ 0.6343           │
  └─────────────────────────────────────────┘

  Best performing model  : XGBoost (AUC = 0.6343)
  Class imbalance fix    : class_weight='balanced' / scale_pos_weight
  Models saved to        : models/ folder

  Proceed to Chapter 4 — Model Evaluation
